# Cuaderno de Ejercicios — Similitud Semántica: spaCy vs. TF-IDF

## Objetivo

Implementar y comparar dos estrategias para calcular similitud entre oraciones: una basada en embeddings de spaCy y otra basada en TF-IDF. El foco está en entender en qué contextos cada enfoque funciona mejor.

## Resultados de aprendizaje

Al terminar este cuaderno vas a poder:

- Implementar una función de similitud semántica usando embeddings de spaCy.
- Implementar la misma función usando TF-IDF y similitud coseno.
- Comparar el comportamiento de ambos métodos ante el mismo corpus.
- Argumentar cuándo conviene usar embeddings y cuándo TF-IDF para similitud de oraciones.

## Relación con cuadernos anteriores

Este cuaderno aplica los conceptos de embeddings (cuadernos 1 a 4) y TF-IDF (cuaderno 5) en una tarea concreta: encontrar la oración más similar a una oración objetivo dentro de una lista.


## Contexto de los ejercicios

El problema que vamos a resolver se llama **búsqueda de similitud de oraciones**: dada una oración objetivo y una lista de oraciones candidatas, queremos encontrar cuál candidata es semánticamente más parecida a la objetivo.

Este problema aparece en aplicaciones reales como:
- Sistemas de preguntas y respuestas (FAQ automáticos).
- Motores de búsqueda semántica.
- Sistemas de recomendación de artículos.

Vamos a resolver el mismo problema con dos enfoques distintos y comparar los resultados.


## Instalaciones e importaciones


In [ ]:
# Instalamos las dependencias necesarias
!pip install spacy scikit-learn -q

# Descargamos el modelo de español de spaCy
# 'es_core_news_md' incluye embeddings de palabras (300 dimensiones)
!python -m spacy download es_core_news_md -q

print("Instalación completada.")


## Ejercicio 1 — Similitud con embeddings de spaCy

### ¿Cómo funciona spaCy para similitud?

spaCy carga un modelo de lenguaje que tiene vectores de palabras incorporados. Cuando procesamos una oración con `nlp(texto)`, spaCy genera un vector para toda la oración promediando los vectores de sus palabras. Luego podemos calcular la similitud entre dos oraciones comparando esos vectores.

### Consigna

Completá la función `encontrar_frase_similar` que:
1. Toma una oración objetivo y una lista de oraciones candidatas.
2. Calcula la similitud coseno entre la oración objetivo y cada candidata.
3. Devuelve la candidata con mayor similitud.

> **Pista:** en spaCy, la similitud entre dos documentos se calcula con `doc1.similarity(doc2)`. El resultado es un número entre 0 y 1.


In [ ]:
import spacy

# Cargamos el modelo de lenguaje en español
# 'es_core_news_md' tiene vectores de palabras de 300 dimensiones
nlp = spacy.load('es_core_news_md')


def encontrar_frase_similar(frase_objetivo, lista_de_frases):
    """
    Encuentra la frase más similar a la frase objetivo dentro de una lista.

    Usa embeddings de spaCy (promedio de vectores de palabras) y
    similitud coseno para comparar las oraciones.

    Args:
        frase_objetivo (str): La oración que queremos comparar.
        lista_de_frases (list): Lista de oraciones candidatas.

    Returns:
        str: La oración de la lista que es más similar a la frase_objetivo.
    """
    # Procesamos la frase objetivo con spaCy para obtener su vector
    doc_objetivo = nlp(frase_objetivo)

    # Guardamos la mejor similitud encontrada y la frase correspondiente
    mejor_similitud = -1
    frase_mas_similar = None

    # Recorremos cada candidata y calculamos su similitud con la objetivo
    for i in range(len(lista_de_frases)):
        frase_candidata = lista_de_frases[i]

        # Procesamos la candidata con spaCy
        doc_candidata = nlp(frase_candidata)

        # Calculamos la similitud coseno entre los dos documentos
        similitud = doc_objetivo.similarity(doc_candidata)

        # Guardamos si es la mejor hasta ahora
        if similitud > mejor_similitud:
            mejor_similitud = similitud
            frase_mas_similar = frase_candidata

    return frase_mas_similar


### Probando la función

Vamos a probar con un ejemplo simple donde la respuesta es bastante clara.

> **Para mirar:** ¿la función devuelve la oración que esperabas? ¿Por qué creés que "helado de vainilla" es más similar a "helado de chocolate" que "ensalada de pepino"?


In [ ]:
# Prueba simple
frase_objetivo_prueba = 'amo el helado de chocolate'
candidatas_prueba = [
    'amo el helado de vainilla',
    'amo la ensalada de pepino'
]

resultado_prueba = encontrar_frase_similar(frase_objetivo_prueba, candidatas_prueba)

print(f"Oración objetivo: '{frase_objetivo_prueba}'")
print(f"Candidatas: {candidatas_prueba}")
print(f"Más similar según spaCy: '{resultado_prueba}'")


## Ejercicio 2 — Similitud con TF-IDF

### ¿Cómo funciona TF-IDF para similitud?

En lugar de usar embeddings semánticos, usamos TF-IDF para vectorizar las oraciones. Cada oración se convierte en un vector de pesos TF-IDF, y luego calculamos la similitud coseno entre la oración objetivo y cada candidata.

La diferencia clave con spaCy es que TF-IDF trabaja con **palabras exactas**: no tiene noción de que "chocolate" y "vainilla" son tipos de helado.

### Consigna

Completá la función `encontrar_frase_similar_tfidf` que hace lo mismo que la del ejercicio 1, pero usando TF-IDF en lugar de spaCy.

> **Pista:** para calcular similitud coseno con sklearn podés usar `cosine_similarity(vector_a, vector_b)`. El truco es vectorizar la frase objetivo junto con las candidatas para que el vocabulario sea consistente.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def encontrar_frase_similar_tfidf(frase_objetivo, lista_de_frases_input):
    """
    Encuentra la frase más similar a la frase objetivo dentro de una lista.

    Usa TF-IDF para vectorizar las oraciones y similitud coseno para compararlas.
    La búsqueda se basa en palabras exactas compartidas, no en significado semántico.

    Args:
        frase_objetivo (str): La oración que queremos comparar.
        lista_de_frases_input (list): Lista de oraciones candidatas.

    Returns:
        str: La oración de la lista que es más similar a la frase_objetivo.
    """
    # Armamos una lista con todas las oraciones juntas
    # La primera es la objetivo; el resto son las candidatas
    todas_las_frases = [frase_objetivo] + lista_de_frases_input

    # Creamos el vectorizador TF-IDF
    vectorizador = TfidfVectorizer()

    # Vectorizamos todas las frases juntas para que compartan el mismo vocabulario
    matriz_tfidf = vectorizador.fit_transform(todas_las_frases)

    # Separamos el vector de la frase objetivo (primera fila) del resto
    vector_objetivo = matriz_tfidf[0]
    vectores_candidatas = matriz_tfidf[1:]

    # Calculamos la similitud coseno entre la objetivo y cada candidata
    similitudes = cosine_similarity(vector_objetivo, vectores_candidatas)

    # 'similitudes' es una matriz de 1 fila; tomamos la primera (y única) fila
    fila_similitudes = similitudes[0]

    # Encontramos el índice de la candidata con mayor similitud
    indice_mejor = fila_similitudes.argmax()

    # Devolvemos la frase candidata correspondiente a ese índice
    frase_ganadora = lista_de_frases_input[indice_mejor]

    return frase_ganadora


### Probando la función TF-IDF

Probamos con el mismo ejemplo del Ejercicio 1 para comparar los resultados.

> **Para mirar:** ¿el resultado es el mismo que con spaCy? ¿Por qué? ¿Qué palabras exactas comparten las oraciones?


In [ ]:
resultado_tfidf = encontrar_frase_similar_tfidf(frase_objetivo_prueba, candidatas_prueba)

print(f"Oración objetivo: '{frase_objetivo_prueba}'")
print(f"Candidatas: {candidatas_prueba}")
print(f"Más similar según TF-IDF: '{resultado_tfidf}'")


## Ejercicio 3 — Comparación con un dataset más desafiante

Ahora vamos a probar ambas funciones con un conjunto de oraciones donde la diferencia entre los dos enfoques se vuelve más visible.

El dataset incluye oraciones sobre fútbol, con variaciones semánticas que TF-IDF puede no capturar bien (por ejemplo, "soccer" vs. "fútbol").

### Preguntas de andamiaje

Antes de ejecutar el código, pensá:

1. ¿Qué hace la función cuando dos oraciones comparten muchas palabras pero tienen significados distintos?
2. ¿Qué hace cuando tienen significados similares pero usan palabras distintas?
3. ¿En cuál de esos dos casos esperás que spaCy funcione mejor? ¿Y TF-IDF?


In [ ]:
# Oración objetivo
oracion_objetivo = "Me gusta mucho el fútbol, mi equipo favorito es River Plate."

# Dataset de candidatas con distintas relaciones semánticas
dataset = [
    "A él también le gusta mucho el fútbol, siempre lo está viendo.",
    "El deporte favorito de María es el básquet, es fanática de San Lorenzo.",
    "El fútbol es el deporte más popular en Argentina.",
    "River Plate es un club histórico de Buenos Aires.",
    "Me apasiona el soccer, soy hincha del Barcelona.",
    "No me interesan los deportes, prefiero leer.",
    "El equipo de River ganó el último clásico.",
]

# Mostramos el dataset numerado
print("Dataset de candidatas:")
print(f"Objetivo: '{oracion_objetivo}'")
print()
for i in range(len(dataset)):
    numero = i + 1
    texto = dataset[i]
    print(f"  [{numero}] {texto}")


### Ejecutando ambas funciones

> **Para mirar:** ¿devuelven la misma oración? Si no, ¿cuál te parece más apropiada y por qué?


In [ ]:
resultado_tfidf_dataset = encontrar_frase_similar_tfidf(oracion_objetivo, dataset)
print(f"Resultado TF-IDF: '{resultado_tfidf_dataset}'")


In [ ]:
resultado_spacy_dataset = encontrar_frase_similar(oracion_objetivo, dataset)
print(f"Resultado spaCy: '{resultado_spacy_dataset}'")


### Reflexión final

Respondé las siguientes preguntas en tu cuaderno o en la celda de texto a continuación:

1. ¿Los dos métodos coincidieron en el resultado o dieron respuestas distintas?
2. ¿Cuál te parece más acertado para este caso? ¿Por qué?
3. ¿En qué situación usarías TF-IDF en lugar de embeddings para similitud?
4. ¿En qué situación usarías embeddings?


*[Escribí tu respuesta acá]*


## Guía de estudio — Similitud de oraciones

## Preguntas y Respuestas Clave

### **Comparación de Enfoques**

**P: ¿Cuál es la diferencia principal entre spaCy y TF-IDF para similitud de oraciones?**  
R: spaCy usa embeddings pre-entrenados que capturan semántica global. TF-IDF solo considera frecuencia de palabras exactas sin contexto semántico.

**P: ¿Por qué spaCy puede encontrar similitud entre sinónimos?**  
R: Sus embeddings están entrenados para que palabras con significados similares tengan vectores cercanos, mientras que TF-IDF trata "coche" y "auto" como completamente diferentes.

**P: ¿En qué casos TF-IDF podría superar a spaCy?**  
R: Cuando necesitas similitud lexical exacta, tienes vocabulario muy específico del dominio, o cuando las palabras clave literales son más importantes que el significado.

### **Implementación Técnica**

**P: ¿Por qué en TF-IDF agregamos la oración objetivo a la lista?**  
R: Para que el vectorizador aprenda el vocabulario completo y pueda comparar todas las oraciones en el mismo espacio vectorial.

**P: ¿Qué hace `argsort()[0][-2]` en la función TF-IDF?**  
R: Obtiene el índice del segundo valor más alto de similitud (excluye la oración objetivo que sería la más similar a sí misma).

**P: ¿Cómo procesa spaCy una oración completa?**  
R: Combina los embeddings de palabras individuales (típicamente promediando) para crear un vector representativo de toda la oración.

### **Análisis de Rendimiento**

**P: ¿Qué patrón observas en el dataset de fútbol entre ambos métodos?**  
R: spaCy encuentra mejor similitud semántica profunda. TF-IDF se enfoca en palabras compartidas literalmente ("fútbol", "River Plate").

**P: ¿Cuándo usarías cada método en una aplicación real?**  
R: spaCy para chatbots, búsqueda semántica, recomendaciones. TF-IDF para búsqueda de documentos, keywords matching, clasificación por temas específicos.

### **Casos de Uso Prácticos**

**P: ¿Cómo mejorarías la función de spaCy para mayor precisión?**  
R: Usar modelos más grandes (es_core_news_lg), preprocesar texto, considerar pesos por importancia de palabras, o usar embeddings especializados del dominio.

**P: ¿Qué limitaciones tiene cada approach?**  
R: spaCy: dependiente del modelo pre-entrenado, puede perder especificidad del dominio. TF-IDF: ignora semántica, sensible al vocabulario exacto.

## Puntos Clave para Recordar

1. **spaCy = semántica global** vs **TF-IDF = frecuencia local**
2. **Embeddings capturan significado** más allá de palabras exactas
3. **TF-IDF mejor para matching literal** y vocabulario específico
4. **Preprocessing afecta ambos métodos** pero diferentemente
5. **Elección depende del caso de uso** específico
6. **Combinar ambos approaches** puede ser óptimo en algunos casos

## Consideraciones Importantes

- spaCy requiere modelos pre-descargados y más memoria
- TF-IDF es más rápido y liviano pero menos "inteligente"
- La calidad del modelo spaCy afecta directamente resultados
- TF-IDF puede funcionar mejor con preprocesamiento agresivo
- Ambos son sensibles a la longitud de las oraciones

## Conexión con Próxima Clase

Estas técnicas de similitud son la base para **aplicaciones avanzadas**: sistemas de recomendación, búsqueda semántica, y extracción de información estructurada de textos.

---
*Consejo: Prueba ambos métodos con tus propias oraciones en español argentino. ¿Cuál maneja mejor lunfardo y referencias culturales locales?*

## Próximos pasos

Con estos ejercicios completamos la exploración práctica de las representaciones semánticas. En el módulo siguiente vamos a ver técnicas más avanzadas de recuperación de información y búsqueda semántica que combinan embeddings con estructuras de índice eficientes.

Para profundizar, te recomendamos revisar el cuaderno anexo (`07_anexo_clasificacion_texto`) donde usamos estas representaciones como entrada para modelos de clasificación supervisada.
